In [2]:
# import os
# import sys

# import requests  # For making HTTP requests
# import pandas as pd  # For data manipulation
# import sqlalchemy  # For interacting with databases
# from sqlalchemy.engine import URL, create_engine
# from shapely import wkb
# import numpy as np  # For numerical operations (if needed)
# import matplotlib.pyplot as plt  # For visualizing data (if needed)
# import seaborn as sns  # For data visualization with Seaborn (if needed)
# import plotly.express as px  # For data visualization with Plotly (if needed)
# import geopandas as gpd
# from matplotlib.dates import HourLocator, DateFormatter

# # Custom imports
# # Add the path to the directory containing your local module
# module_path = os.path.abspath(os.path.join('..', '/Users/philipmac2/Development/semester_thesis/repo_philip/sa_philip_rosborough')) 
# if module_path not in sys.path:
#     sys.path.append(module_path)


# from utils.utilities import run_sql, run_sql_query  # For custom SQL query execution
# from utils.config import DBConfig

"""
This script is responsible for data acquisition by filtering and processing data 
from an SQL database using predefined queries. The retrieved data is then used 
for further calculations in subsequent processing steps.
"""

import os
import sys
import subprocess
from pathlib import Path
from datetime import datetime

# Third-party libraries
import requests  # For making HTTP requests
import pandas as pd  # For data manipulation
import sqlalchemy  # For database interaction
from sqlalchemy.engine import URL, create_engine
from shapely import wkb  # For geospatial data processing

# This is used to cache function return values (SQL query text → return value)
from joblib import Memory 

# Custom code and numerical/visualization tools
import numpy as np  # For numerical operations
import matplotlib.pyplot as plt  # For data visualization
import seaborn as sns  # For Seaborn-based plots
import plotly.express as px  # For interactive visualizations using Plotly
import geopandas as gpd  # For geospatial data analysis
from matplotlib.dates import HourLocator, DateFormatter
from utilities import run_sql  # Custom SQL execution utility

# Determine module path (handles Jupyter Notebook and Python script cases)
try:
    MODULE_PATH = Path(__file__).resolve().parent.parent  # Standard for scripts
    SCRIPT_PATH = Path(__file__).resolve()
except NameError:
    MODULE_PATH = Path.cwd().parent  # Jupyter Notebook case
    SCRIPT_PATH = MODULE_PATH / "data_acquisition.ipynb"  # Default fallback for Jupyter

if MODULE_PATH.as_posix() not in sys.path:
    sys.path.append(MODULE_PATH.as_posix())

# Change working directory to project root dynamically
PROJECT_ROOT = MODULE_PATH  # Assuming repo root is two levels up
os.chdir(PROJECT_ROOT)

# Setup caching for SQL queries (cache directory: `.cache/`)
CACHE_DIR = PROJECT_ROOT / ".cache"
CACHE_DIR.mkdir(exist_ok=True)  # Ensure cache directory exists
memory = Memory(CACHE_DIR, verbose=0)


@memory.cache
def cached_run_sql(query: str, version: int = 1):  # Increment version when query changes
    """
    Executes an SQL query and caches the result.

    :param query: SQL query string
    :param version: Version number to invalidate cache if needed
    :return: Query result (cached)
    """
    return run_sql(query)




ModuleNotFoundError: No module named 'config'

In [ ]:
sys.path = ['/Users/philipmac2/Development/semester_thesis/repo_philip/sa_philip_rosborough']
# cdir = os.getcwd()
# python_root = os.path.abspath(cdir)
python_root = '/Users/philipmac2/Development/semester_thesis/repo_philip/sa_philip_rosborough'

sys.path.append(os.path.join(python_root, "utils"))

conf_path = os.path.join(python_root, "config", "db.conf")
conf_template_path = os.path.join(python_root, "config", "db_template.conf")

dbconfig = DBConfig(conf_path, conf_template_path)

con_url = URL.create(
    drivername="postgresql",
    host=dbconfig.host,
    port=dbconfig.port,
    username=dbconfig.user,
    password=dbconfig.password,
    database=dbconfig.name
)

engine = create_engine(con_url, echo=True, pool_pre_ping=True)
connection = engine.raw_connection()
cursor = connection.cursor()

2025-02-11 17:41:48,242 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2025-02-11 17:41:48,243 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-02-11 17:41:48,357 INFO sqlalchemy.engine.Engine select current_schema()
2025-02-11 17:41:48,358 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-02-11 17:41:48,472 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2025-02-11 17:41:48,473 INFO sqlalchemy.engine.Engine [raw sql] {}


In [ ]:
#TODO delete
#os.chdir('/Users/philipmac2/Development/semester_thesis/repo_philip/sa_philip_rosborough')

# trips are stored in anon_master (this is just a join of (materialized) views anon_*
df_trips = run_sql('trips', index_col='track_id')

# TODO update to add freight forwarder number based on vehicle_id:
df_trips['freight_forwarder'] = df_trips['vehicle_id'].apply(
    lambda x: 1 if x <= 10 else (2 if x <= 20 else (3 if x <= 30 else 4))
    )

df_trips.to_csv('input/stations/tracks.csv')
df_trips

,vehicle_id,tour_id,start_time,stop_time,distance,track_gap,avg_speed,max_speed,n_signal_loss,d_signal_loss,r_signal_loss,avg_hdop,home_base,long_haul,rest_area,service_area_fuel,industrial_area,cid,freight_forwarder
track_id,,,,,,,,,,,,,,,,,,,
0,1,0,2021-09-10 11:04:53.598000+02:00,2021-09-10 12:07:57.395000+02:00,30751.0,NaN,8.13,20.58,0.0,0.0,0.000000,0.612354,False,False,False,False,True,149,1
1,1,0,2021-09-10 12:57:54.718000+02:00,2021-09-10 13:13:53.221000+02:00,7762.0,37.504057,8.10,21.79,0.0,0.0,0.000000,1.230128,False,False,False,False,True,-1,1
2,1,0,2021-09-10 13:25:20.146000+02:00,2021-09-10 14:09:50.802000+02:00,19432.0,4.895394,7.28,24.75,0.0,0.0,0.000000,0.549358,True,False,False,False,True,0,1
3,1,1,2021-09-10 14:21:35.547000+02:00,2021-09-10 14:25:26.391000+02:00,56.0,3.836167,0.24,2.44,0.0,0.0,0.000000,0.852446,True,False,False,False,True,0,1
4,1,1,2021-09-10 15:39:48.925000+02:00,2021-09-10 15:45:05.392000+02:00,201.0,12.816326,0.64,3.80,0.0,0.0,0.000000,0.656429,True,False,False,False,True,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151593,163,18999,2024-01-05 15:43:20.638000+01:00,2024-01-05 15:43:29.893000+01:00,48.0,399.688607,5.19,11.93,37.0,0.0,0.000000,5.788163,False,False,False,False,False,825,4
151594,163,18999,2024-01-08 16:09:28.961000+01:00,2024-01-08 16:24:28.879000+01:00,4985.0,55070.176671,5.54,22.39,18.0,5.0,0.001003,1.310957,True,False,False,False,False,809,4
151595,163,19000,2024-01-09 18:04:37.533000+01:00,2024-01-09 18:05:44.141000+01:00,71.0,44965.860287,1.07,5.66,13.0,0.0,0.000000,2.008971,False,False,False,False,True,928,4


In [ ]:
# fleet information (especially fleet id)
df_fleet = run_sql('fleet', index_col='vehicle_id')
df_fleet.to_csv('input/home/fleet.csv')
df_fleet.head()

,fleet_test_id,gross_vehicle_weight,total_mass_with_trailer,axle_class
vehicle_id,,,,
1,1,18000.0,40000.0,98.0
2,1,18000.0,40000.0,98.0
3,1,18000.0,40000.0,98.0
4,1,18000.0,40000.0,98.0
5,1,18000.0,40000.0,98.0


In [ ]:
# track_id map information

#alternative shorter version, without checking if index and track_id_new match
#df_track_ids = run_sql('track_ids', index_col='track_id_new')

df_track_ids = run_sql('track_ids')
assert all(df_track_ids.index == df_track_ids['track_id_new']), 'Index does not match track_id_new'
df_track_ids.set_index('track_id_new', inplace=True)

df_track_ids.to_csv('input/stations/track_ids.csv')
df_track_ids.head()

,track_id
track_id_new,
0,1500000356539
1,1500000360362
2,1500000370891
3,1500000374256
4,1500000359843


In [ ]:
# TODO maybe implement the collection of the reduced tracks for energy simulation here

# SQL query to get the detailed GPS data for a specific track_id
# select * from ftm_gps_lon_lat_from_track_id ({trackid})

detailed_tracks = run_sql_query('select * from ftm_gps_lon_lat_from_track_id (1500000552337)')

#track_gps = run_sql_query('select * from ftm_gps_lon_lat_from_track_id(25)', index_col="track_id")

________________________________________________________________________________
[Memory] Calling utils.utilities.run_sql_query...
run_sql_query('select * from ftm_gps_lon_lat_from_track_id (1500000552337)')
SQL query: 
select * from ftm_gps_lon_lat_from_track_id (1500000552337)


ProgrammingError: (psycopg2.errors.InsufficientPrivilege) FEHLER:  keine Berechtigung für Schema track
LINE 6:  track.track
         ^
QUERY:  SELECT
	track.track.vehicle_id,
	start_time,
	stop_time
                                             FROM
	track.track
WHERE id=_track_id
CONTEXT:  PL/pgSQL-Funktion ftm_gps_lon_lat_from_track_id(bigint) Zeile 7 bei SQL-Anweisung

[SQL: select * from ftm_gps_lon_lat_from_track_id (1500000552337)]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [ ]:
 # trips are stored in anon_master (this is just a join of (materialized) views anon_*
df_nuts3_trips = pd.read_sql('''select t.fid, n, ST_Transform(nm.geom, 3857) as geom from 
(
select substring(anz.fid,0,6) as fid, sum(count) as n from nefton_rio_telemetry.anon_nuts_zones anz
group by substring(anz.fid,0,6)
order by n desc
) t
join administrative_boundaries.nuts_2021_10m nm on t.fid  = nm.fid
where n > 0
order by n desc''', con=engine, index_col='fid')
# TODO Why is the db nefton_rio_telemetry used here? Shouldn't it be spirite? Should I do the same?
# sql script for the creation of nefton_rio_telemetry is in old repo on LRZ syncandshare

df_nuts0 = pd.read_sql('''select gid, fid, ST_Transform(nm.geom, 3857) as geom from 
administrative_boundaries.nuts_2021_10m nm where levl_code = 0''', con=engine, index_col='gid')

df_nuts1 = pd.read_sql('''select gid, fid, ST_Transform(nm.geom, 3857) as geom from 
administrative_boundaries.nuts_2021_10m nm where levl_code = 1''', con=engine, index_col='gid')

gdf_nuts3_trips = gpd.GeoDataFrame(df_nuts3_trips[['n']], geometry=df_nuts3_trips.geom.apply(lambda x: wkb.loads(x, hex=True)))

gdf_nuts0 = gpd.GeoDataFrame(df_nuts0[['fid']], geometry=df_nuts0.geom.apply(lambda x: wkb.loads(x, hex=True)))
gdf_nuts1 = gpd.GeoDataFrame(df_nuts1[['fid']], geometry=df_nuts1.geom.apply(lambda x: wkb.loads(x, hex=True)))



#TODO: Für Heatmap nutzen und Gedanken dazu machen, wie man das anonymisieren könnte

# locations - not to be published only for the heat map!
df_start_locations = pd.read_sql('SELECT track_id, start_location from spirite.anon_layer_base', con=engine, index_col='track_id')
df_start_locations['start_location'] = df_start_locations.start_location.apply(lambda x: wkb.loads(x, hex=True))
df_start_locations.to_pickle('input/df_start_locations.pkl')
df_start_locations.head()

2025-01-08 17:08:35,023 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-01-08 17:08:35,024 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s
2025-01-08 17:08:35,024 INFO sqlalchemy.engine.Engine [generated in 0.00079s] {'table_name': 'select t.fid, n, ST_Transform(nm.geom, 3857) as geom from \n(\nselect substring(anz.fid,0,6) as fid, sum(count) as n from nefton_rio_telemetry.anon_n ... (17 characters truncated) ... oup by substring(anz.fid,0,6)\norder by n desc\n) t\njoin administrative_boundaries.nuts_2021_10m nm on t.fid  = nm.fid\nwhere n > 0\norder by n desc', 'param_1': 'r

ProgrammingError: (psycopg2.errors.UndefinedTable) FEHLER:  Relation »nefton_rio_telemetry.anon_nuts_zones« existiert nicht
LINE 3: ...bstring(anz.fid,0,6) as fid, sum(count) as n from nefton_rio...
                                                             ^

[SQL: select t.fid, n, ST_Transform(nm.geom, 3857) as geom from 
(
select substring(anz.fid,0,6) as fid, sum(count) as n from nefton_rio_telemetry.anon_nuts_zones anz
group by substring(anz.fid,0,6)
order by n desc
) t
join administrative_boundaries.nuts_2021_10m nm on t.fid  = nm.fid
where n > 0
order by n desc]
(Background on this error at: https://sqlalche.me/e/20/f405)